# S10 · A credit-default model, with a bias audit

This is the notebook that matters most. We build a credit-default classifier, the
kind a bank or a lending app really uses, and then we do the part that too many
teams skip: we **audit it for fairness**. Does it approve one group of people less
often than another? Does it make more mistakes for one group? We check, in a
table and a chart, and we write down honestly what we find.

The bias-audit part is a required, graded piece of the course project.

**New here? Read this once.**

- New to Python? Run each cell top to bottom and read the note above it. The audit
  is just counting and comparing, and every step is explained in plain words.
- New to the idea of a "bias audit"? It is simply: split people into groups and
  check whether the model treats the groups differently. That is the whole idea.
- Already confident? Look for the cell marked **Stretch (optional)** near the end.
- Stuck on a word? It is in `primers/glossary.md`.

## Setup

Run the next cell to load the libraries and the dataset. It works in **Google
Colab** and **on your own machine**.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


def load(name):
    """Load one of this session's teaching datasets."""
    local = Path("../data") / f"{name}.csv"
    if local.exists():
        return pd.read_csv(local)
    return pd.read_csv(f"/content/{name}.csv")


print("Setup complete.")

## Step 1 — meet the credit dataset

This is a synthetic credit dataset: 3000 past applicants, each with realistic
inputs (income, credit score, existing loans, age, loan amount) **and** a
sensitive attribute we will audit against: `gender`, coded `0` or `1`.

Two things to hold on to. First, in this made-up world the two genders are
**equally creditworthy on average** — so any gap the model shows later is a real
problem, not a genuine difference between people. Second, look at the
`recorded_income` column: group 0's recorded income has been shaved down, a quiet
data-collection flaw we will expose in Step 3.

In [ ]:
data = load("credit_bias_data")
data.head()

## Step 2 — check the two groups really are equally creditworthy

Before we blame the model for any gap, confirm the ground truth is fair. The
**default rate** (the true answer) should be about the same for both genders.
Only then does a difference in the model's decisions count as bias.

In [ ]:
default_rate = data.groupby("gender")["default"].mean()

print("true default rate, group 0:", round(default_rate[0], 3))
print("true default rate, group 1:", round(default_rate[1], 3))
print("\nThe two groups default at about the same rate -> equally creditworthy.")

## Step 3 — find the data flaw

Real datasets are collected by imperfect processes. Here, group 0's *recorded*
income has been under-recorded in the database (a measurement problem). Their
true creditworthiness is unchanged; only the number in the file is wrong.

We never tell the model about gender. But because the recorded income is now lower
for group 0, income has quietly become a **proxy** for gender: a stand-in that
carries the same information. This is exactly how bias sneaks into a model
sideways.

In [ ]:
income_mean = data.groupby("gender")["recorded_income"].mean()

print("average recorded income, group 0:", round(income_mean[0], 1))
print("average recorded income, group 1:", round(income_mean[1], 1))
print("\nGroup 0 looks poorer in the data, even though it is not.")

## Step 4 — assemble the table the model will see

We put the inputs into a pandas table. Notice what is **not** here: gender. Leaving
the sensitive column out is a common first attempt at fairness. We will see it is
not enough. We keep gender aside separately, used only for the audit.

In [ ]:
# The inputs the model is allowed to use. Note: no gender column.
X = data[["recorded_income", "credit_score", "existing_loans", "age", "loan_amount"]]
y = data["default"]

# Kept aside for the audit only. The model never sees this.
gender = data["gender"]

print("Columns the model can use:", list(X.columns))
X.head()

## Step 5 — split into training and test sets

We split the inputs, the answer, **and** the gender column together, so that each
test applicant still has their gender label attached for the audit afterwards.

In [ ]:
from sklearn.model_selection import train_test_split

(X_train, X_test,
 y_train, y_test,
 gender_train, gender_test) = train_test_split(
    X, y, gender,
    test_size=0.3, random_state=42)

print("training applicants:", X_train.shape[0])
print("test applicants    :", X_test.shape[0])

## Step 6 — train a classifier to predict default

We use a Random Forest, the reliable workhorse from the previous notebook. It
learns only from the columns in `X_train` (so, not gender).

In [ ]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(n_estimators=200, random_state=42)
model.fit(X_train, y_train)

test_accuracy = model.score(X_test, y_test)
print("test accuracy:", round(test_accuracy, 3))

## Step 7 — turn predictions into approve / reject decisions

For the audit we frame the model's output as a lending decision: **approve** the
loan when the model predicts the applicant will **repay** (that is, predicts 'not
default'). So we compute one approve-or-reject decision per test applicant.

In [ ]:
predicted_default = model.predict(X_test)

# We APPROVE the loan when the model predicts the person will NOT default.
approved = (predicted_default == 0).astype(int)

print("approved (1) vs rejected (0) on the test set:", np.bincount(approved))

## Step 8 — a small helper to report on any group

The audit repeats the same three measurements for different groups, so we write
one small helper and reuse it. Given a chosen set of test applicants (a
true/false mask), it returns:

- the **approval rate**: what fraction of that group we approved.
- the **false-rejection rate**: of the people in that group who *would* have
  repaid, what fraction did we wrongly reject. A wrongly-rejected good customer is
  a lost customer, and unfair if it hits one group harder.
- the **false-approval rate**: of the people who *did* default, what fraction did
  we wrongly approve. That is money the bank loses.

In [ ]:
truth = np.array(y_test)

def group_report(in_group):
    approvals_here = approved[in_group]
    truth_here = truth[in_group]

    approval_rate = approvals_here.mean()

    good_applicants = (truth_here == 0)
    false_rejection_rate = (approvals_here[good_applicants] == 0).mean()

    bad_applicants = (truth_here == 1)
    false_approval_rate = (approvals_here[bad_applicants] == 1).mean()

    return approval_rate, false_rejection_rate, false_approval_rate

print("Helper ready.")

## Step 9 — the bias audit: approval and error rates by gender

Now the core check. We split the test applicants by gender and run the helper on
each group. Remember: in our world the two groups are equally creditworthy, so a
fair model would give them very similar numbers.

In [ ]:
approval_0, false_reject_0, false_approve_0 = group_report(gender_test == 0)
approval_1, false_reject_1, false_approve_1 = group_report(gender_test == 1)

print("group 0: approval rate =", round(approval_0, 3),
      " false-rejection =", round(false_reject_0, 3),
      " false-approval =", round(false_approve_0, 3))
print("group 1: approval rate =", round(approval_1, 3),
      " false-rejection =", round(false_reject_1, 3),
      " false-approval =", round(false_approve_1, 3))
print("\napproval-rate gap (group1 - group0):",
      round(approval_1 - approval_0, 3))

## Step 10 — the audit table

We collect the per-group numbers into one small table, the kind you would paste
straight into a fairness report.

In [ ]:
audit_table = pd.DataFrame({
    "group": ["group 0", "group 1"],
    "approval_rate": [approval_0, approval_1],
    "false_rejection_rate": [false_reject_0, false_reject_1],
    "false_approval_rate": [false_approve_0, false_approve_1],
})

audit_table = audit_table.round(3)
print(audit_table.to_string(index=False))

## Step 11 — the audit chart

A bar chart makes the gap obvious at a glance. This is the single most useful
fairness picture, and the kind of chart we ask for in the project report.

In [ ]:
groups = ["group 0", "group 1"]
rates = [approval_0, approval_1]

plt.figure(figsize=(6, 5))
bars = plt.bar(groups, rates, color=["#C0392B", "#2E75B6"])

for one_bar, rate in zip(bars, rates):
    plt.text(one_bar.get_x() + one_bar.get_width() / 2,
             rate + 0.01,
             str(round(rate * 100, 1)) + "%",
             ha="center")

plt.ylabel("approval rate")
plt.ylim(0, 1)
plt.title("Bias audit: approval rate by gender")
plt.show()

## Step 12 — audit a second grouping: age band

Gender is not the only group worth checking. A model can also treat younger and
older applicants differently, which matters for lending. Because our helper works
on any group, checking age is now easy: we split the test applicants into "under
35" and "35 and over" and run the same report. Age *was* an input to the model
here, so this asks a fair question: are the model's decisions lopsided by age?

In [ ]:
age_test = X_test["age"].values

younger = (age_test < 35)
older = (age_test >= 35)

approval_young, false_reject_young, false_approve_young = group_report(younger)
approval_old, false_reject_old, false_approve_old = group_report(older)

print("under 35 : approval rate =", round(approval_young, 3),
      " false-rejection =", round(false_reject_young, 3),
      " false-approval =", round(false_approve_young, 3))
print("35 & over: approval rate =", round(approval_old, 3),
      " false-rejection =", round(false_reject_old, 3),
      " false-approval =", round(false_approve_old, 3))
print("\napproval-rate gap (older - younger):",
      round(approval_old - approval_young, 3))

### Stretch (optional) — a pin-code that leaks gender

Skip this if you are new to code. For the curious, here is proxy bias in its most
famous form: **pin-code**. A locality often lines up with who lives there, so a
pin-code column can quietly stand in for a protected attribute, even when you
never included that attribute.

We invent a pin-code "area" that is correlated with gender (group 0 mostly lands
in one area), add it as a model input, retrain, and watch what happens to the
gender approval gap.

In [ ]:
np.random.seed(42)
number_of_people = len(data)

usually_matches = np.random.uniform(0, 1, size=number_of_people) < 0.85
pincode_area = np.where(usually_matches, gender, 1 - gender)

pincode_train, pincode_test = train_test_split(
    pincode_area, test_size=0.3, random_state=42)

X_train_plus = X_train.copy()
X_train_plus["pincode_area"] = pincode_train
X_test_plus = X_test.copy()
X_test_plus["pincode_area"] = pincode_test

model_plus = RandomForestClassifier(n_estimators=200, random_state=42)
model_plus.fit(X_train_plus, y_train)

approved_plus = (model_plus.predict(X_test_plus) == 0).astype(int)

approval_0_plus = approved_plus[gender_test == 0].mean()
approval_1_plus = approved_plus[gender_test == 1].mean()

print("gender approval gap WITHOUT pincode:", round(approval_1 - approval_0, 3))
print("gender approval gap WITH pincode   :",
      round(approval_1_plus - approval_0_plus, 3))
print("\npincode importance in the new model:",
      round(model_plus.feature_importances_[-1], 3))
print("A feature we added with no real credit information still gets used,")
print("because it carries gender. That is a proxy.")

## What we found, and what to do

Read your own numbers above before reading this.

**What we found.** Group 0 is approved noticeably less often than group 1, even
though both groups default at the same rate, and even though we never gave the
model the gender column. So where did the bias come from? From income: the dataset
records group 0's income as lower (Step 3), and income is a strong predictor, so
recorded income silently acted as a **proxy** for gender. The optional pin-code
cell shows the same trap with geography. This is the key lesson: dropping the
sensitive column does not make a model fair if other columns leak the same
information.

**What to do about it.** A few honest options, roughly in order of effort:

1. Fix the data. Find and correct the income-recording problem at the source.
   Fixing the data beats patching the model.
2. Audit and document. Report the gap openly, as we did here. A documented
   disparity is a finding, not a failure to hide.
3. Watch the proxies. Be suspicious of inputs like pin-code or recorded income
   that can stand in for a protected attribute.
4. Use fairness-aware methods (named only here): re-weighting the data, or adding
   a fairness constraint during training.

**The takeaway for your project.** A strong accuracy score is not enough. You must
audit *outcomes* across groups, gender and age band and any proxy like pin-code,
not just check which columns you fed in, and you must write down honestly what you
found. That written bias-audit sub-section is a required, graded part of the
project.